In [1]:
import numpy as np
import pandas as pd

customers_path = "/content/synthetic_bank_customers.csv"
products_path = "/content/banking_products.csv"

customers_df = pd.read_csv(customers_path)
products_df = pd.read_csv(products_path)

print("Number of Customers: ", len(customers_df))
display("Products: ", products_df)

Number of Customers:  25000


'Products: '

,product_id,product_name,product_type,interest_rate,min_balance,max_balance,fees,currency
0,1,Basic Savings Account,Deposit,1.5,50,NaN,10,TTD
1,2,Premium Savings Account,Deposit,2.0,5000,NaN,20,TTD
2,3,Checking Account,Deposit,0.5,0,NaN,15,TTD
3,4,Personal Loan,Loan,10.0,1000,200000.0,0,TTD
4,5,Home Loan,Loan,7.0,50000,1000000.0,0,TTD
5,6,Credit Card,Credit,18.0,0,50000.0,50,TTD
6,7,Investment Account,Investment,5.0,1000,NaN,0,TTD
7,8,Car Loan,Loan,8.0,5000,500000.0,0,TTD
8,9,Retirement Account,Deposit,3.5,1000,NaN,0,TTD


In [2]:
import pandas as pd
import numpy as np

# -----------------------------
# Number of reviews
# -----------------------------
num_reviews = 75000

# -----------------------------
# Review text pools (for sentiment)
# -----------------------------
positive_reviews = [
    "Great product",
    "Very satisfied",
    "Excellent service",
    "Highly recommend",
    "Works perfectly",
    "Very convenient",
    "Good interest rate",
    "Easy to use",
    "Very reliable",
    "Happy with this product"
]

neutral_reviews = [
    "It is okay",
    "Average experience",
    "Nothing special",
    "Works as expected",
    "Decent product",
    "No major issues",
    "Service was fine",
    "Acceptable overall",
    "Not bad",
    "Fair product"
]

negative_reviews = [
    "Very disappointing",
    "Bad experience",
    "Not satisfied",
    "Too many fees",
    "Customer service was poor",
    "Would not recommend",
    "Difficult to use",
    "Interest rate too high",
    "Not worth it",
    "Very frustrating"
]

# -----------------------------
# Customer and product selection
# -----------------------------
review_customer_ids = np.random.choice(customers_df['customer_id'], num_reviews)
review_product_ids = np.random.choice(products_df['product_id'], num_reviews)

# -----------------------------
# Review dates (3 years)
# -----------------------------
review_dates = pd.to_datetime("2023-01-01") + pd.to_timedelta(
    np.random.randint(0, 1095, num_reviews), unit="d"
)

# -----------------------------
# Ratings distribution
# -----------------------------
ratings = np.random.choice(
    [1,2,3,4,5],
    num_reviews,
    p=[0.1, 0.15, 0.25, 0.3, 0.2]   # realistic rating distribution
)

# -----------------------------
# Generate review text based on rating
# -----------------------------
reviews = []

for r in ratings:

    if r >= 4:
        reviews.append(np.random.choice(positive_reviews))

    elif r == 3:
        reviews.append(np.random.choice(neutral_reviews))

    else:
        reviews.append(np.random.choice(negative_reviews))

# -----------------------------
# Create Reviews DataFrame
# -----------------------------
reviews_df = pd.DataFrame({
    "review_id": np.arange(1, num_reviews + 1),
    "customer_id": review_customer_ids,
    "product_id": review_product_ids,
    "review_date": review_dates,
    "review": reviews,
    "rating": ratings
})

display(reviews_df.head())

,review_id,customer_id,product_id,review_date,review,rating
0,1,15984,7,2024-01-09,Easy to use,4
1,2,126,7,2024-08-16,It is okay,3
2,3,24902,9,2023-02-03,Customer service was poor,2
3,4,1603,9,2025-06-22,Not bad,3
4,5,15341,2,2024-11-22,Decent product,3


In [3]:
filename = "customer_reviews.csv"
reviews_df.to_csv(filename, index=False)

In [4]:
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [7]:
df = reviews_df.copy()

sia = SentimentIntensityAnalyzer()

def calculate_sentiment(text):
    sentiment_scores = sia.polarity_scores(text)
    return sentiment_scores['compound']

def catorize_sentiment(score, rating):
    if score >= 0.05:
        if rating >= 4:
            return 'Positive'
        elif rating == 3:
            return 'Mixed Positive'
        else:
            return "Mixed Negative"
    elif score <= -0.05:
        if rating <= 2:
            return 'Negative'
        elif rating == 3:
            return 'Mixed Negative'
        else:
            return "Mixed Positive"
    else:
        if rating >= 4:
            return 'Positive'
        elif rating <= 2:
            return 'Negative'
        else:
            return "Neutral"

def sentiment_bucked(score):
    if score >= 0.05:
        return '0.5 to 1.0'
    elif 0.0 <= score < 0.5:
        return '0.0 to 4.9'
    elif -0.5 <= score < 0.0:
        return '-0.49 to 0.0'
    else:
        return '-1.0 to -0.5'


df['Sentiment Score'] = df['review'].apply(calculate_sentiment)
df['Sentiment Category'] = df.apply(lambda row: catorize_sentiment(row['Sentiment Score'], row['rating']), axis=1)
df['Sentiment Bucket'] = df['Sentiment Score'].apply(sentiment_bucked)

display(df.head())

,review_id,customer_id,product_id,review_date,review,rating,Sentiment Score,Sentiment Category,Sentiment Bucket
0,1,15984,7,2024-01-09,Easy to use,4,0.4404,Positive,0.5 to 1.0
1,2,126,7,2024-08-16,It is okay,3,0.2263,Mixed Positive,0.5 to 1.0
2,3,24902,9,2023-02-03,Customer service was poor,2,-0.4767,Negative,-0.49 to 0.0
3,4,1603,9,2025-06-22,Not bad,3,0.4310,Mixed Positive,0.5 to 1.0
4,5,15341,2,2024-11-22,Decent product,3,0.0000,Neutral,0.0 to 4.9


In [8]:
filename = "customer_reviews_sentiment.csv"
df.to_csv(filename, index=False)